# KCD 2026 · 노트북 하나로 전부 실행

코드 버전 `7358571`. 이 파일 하나에 필요한 코드가 모두 들어 있습니다. 첫 셀의 `MIMIC_ROOT`만 확인하고 위에서부터 끝까지 실행하세요.

기관 PC에서 이 노트북 하나만 위에서부터 끝까지 실행하면 됩니다.

| 단계 | 하는 일 | 대략 |
|---|---|---|
| 1 | 전체 MIMIC-IV로 분석 노트북 실행 (코호트·사건·세 에이전트·평가) | 가장 오래 걸림 (labevents 전체 스캔) |
| 2 | 모델 개발·평가: Agent 3 위험, 24시간 궤적, Agent 2 예후 4경로, Agent 1 원인 8개 0/1, 그룹 Shapley, 전과 후 실험 | 수십 분 |
| 3 | 공개 MIMIC-IV Demo 100명에 학습된 모델을 적용해 시연 화면 생성 (데모 환자는 학습에서 제외됨) | 수 분 |
| 4 | 돌려줄 파일 하나로 묶기: `kcd_return_package_*.zip` | 1분 |

**돌려주는 zip에 들어가는 것**: 집계 결과(`results_full.json`, 10 미만 칸 가림), 논문 표 CSV, 집계 그림 PNG, 공개 데모 환자의 시연 화면 JSON, 실행 보고서.
**들어가지 않는 것**: 학습된 모델 파일, 전체 MIMIC-IV 환자 단위 파일, 실행된 노트북 사본. 모두 이 PC의 `run_all/` 폴더에만 남습니다.

중간에 한 단계가 실패해도 다음 단계와 압축은 계속 진행하고, 실패 내용을 보고서에 적습니다. 그 zip을 그대로 보내주세요.

**다시 실행할 때**: 코드만 바뀐 새 노트북을 받았다면 `RUN_STEPS['full'] = False`로 두면 가장 오래 걸리는 1단계를 건너뛰고 이전 결과(`kcd_work/run_all`)를 재사용합니다.

In [ ]:
# ===== 설정: 필요하면 여기만 고칩니다 =====
from pathlib import Path
import os, sys, json, time, shutil, zipfile, traceback, subprocess, platform
from datetime import datetime

BASE = Path.cwd() / 'kcd_work'         # 코드와 중간 산출물 (이 PC에만 남음)
REPO = BASE / 'code'                   # 아래 셀이 코드를 여기에 풉니다
MIMIC_ROOT = Path(r'\\10.7.12.146\데이터 및 소스\MIMIC\MIMIC4-V3.1')   # hosp/ icu/ 가 있는 폴더
MAX_PATIENTS = None        # 시간이 부족하면 20000 같은 숫자 (None = 전체)
DEMO_CSV = BASE / 'mimic-iv-demo-csv'   # 없으면 3단계에서 자동 생성
DEMO_FHIR_DIR = None       # 인터넷이 막혀 있으면 미리 받은 FHIR 데모의 fhir 폴더 경로
WORK = BASE / 'run_all'    # 환자 단위 중간 산출물 (이 PC에만 남음)
RUN_STEPS = {'full': True, 'develop': True, 'demo': True, 'package': True}
# 기관 LLM (선택). 없으면 Agent 1은 규칙 기준 모델로 실행됩니다. 비밀키는 노트북에 적지 말고 환경변수로.
# os.environ['LLM_BASE_URL'] = 'https://.../v1'; os.environ['LLM_MODEL'] = '...'
REPORT = {'started_at': datetime.now().isoformat(timespec='seconds'), 'steps': {}, 'python': platform.python_version()}
print('REPO', REPO); print('MIMIC_ROOT 접근 가능:', MIMIC_ROOT.exists())
OUTBOX = Path.cwd()   # 돌려줄 zip이 이 노트북 옆에 생깁니다


In [ ]:
# ===== 코드 풀기 (수정하지 마세요) =====
import base64, io, tarfile
PAYLOAD = "..."
if REPO.exists(): shutil.rmtree(REPO)
REPO.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode='r:gz') as t:
    try: t.extractall(REPO, filter='data')
    except TypeError: t.extractall(REPO)
sys.path.insert(0, str(REPO)); WORK.mkdir(parents=True, exist_ok=True)
for m in [k for k in sys.modules if k.split('.')[0] in ('kcd_liver', 'scripts')]: del sys.modules[m]
REPORT['code_version'] = '7358571'
print('코드 버전 7358571:', sum(1 for _ in REPO.rglob('*.py')), '개 파이썬 파일 ->', REPO)


In [ ]:
# ===== 환경 확인 =====
need = ['pandas', 'numpy', 'sklearn', 'matplotlib', 'joblib', 'nbformat', 'nbclient', 'ipykernel']
missing = []
for m in need:
    try: __import__(m)
    except ImportError: missing.append(m)
if missing:
    print('설치가 필요합니다:', missing)
    print('pip install', ' '.join('scikit-learn' if m == 'sklearn' else m for m in missing))
    raise SystemExit('위 패키지를 설치하고 커널을 재시작한 뒤 다시 실행하세요.')
import pandas as pd, numpy as np, sklearn
REPORT['versions'] = {'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__}
print(REPORT['versions'])

In [ ]:
# ===== 도우미: 분석 노트북을 설정만 바꿔 실행 =====
import nbformat
from nbclient import NotebookClient
MAIN_NB = REPO / 'notebooks' / 'KCD_MIMICIV_liver_agents_v2.ipynb'
ORIG_ROOT = "ROOT = Path(r'\\\\10.7.12.146\\데이터 및 소스\\MIMIC\\MIMIC4-V3.1')"

def run_main(name, root, source, max_patients=None, extra=None, env=None):
    work = WORK / name; work.mkdir(parents=True, exist_ok=True)
    nb = nbformat.read(MAIN_NB, as_version=4)
    subs = [(ORIG_ROOT, f"ROOT = Path(r'{root}')"), ("DATA_SOURCE = 'mimic-iv'", f"DATA_SOURCE = '{source}'"),
            ("MAX_PATIENTS = None  # all patients", f"MAX_PATIENTS = {max_patients!r}  # all patients")] + list((extra or {}).items())
    hit = {a: 0 for a, _ in subs}
    for c in nb.cells:
        for a, b in subs:
            if a in c.source: c.source = c.source.replace(a, b); hit[a] += 1
    miss = [a for a, n in hit.items() if n == 0]
    if miss: raise RuntimeError(f'설정 줄을 찾지 못했습니다: {miss}')
    env = {'KCD_REPO': REPO, **(env or {})}
    old = {k: os.environ.get(k) for k in (env or {})}
    os.environ.update({k: str(v) for k, v in (env or {}).items()})
    t = time.time()
    try:
        code_idx = [i for i, cc in enumerate(nb.cells) if cc.cell_type == 'code']
        t_last = [time.time()]
        timing = REPORT.setdefault('cell_minutes', {}).setdefault(name, [])
        def _done(cell, cell_index, execute_reply=None, **kw):
            dt = (time.time() - t_last[0]) / 60; t_last[0] = time.time()
            head = next((l.strip() for l in cell.source.splitlines() if l.strip()), '')[:60]
            k = code_idx.index(cell_index) + 1 if cell_index in code_idx else cell_index
            timing.append({'cell': cell_index, 'minutes': round(dt, 2), 'head': head})
            print(f'[{name}] {k}/{len(code_idx)} 완료 ({dt:.1f}분, 누적 {(time.time() - t) / 60:.1f}분): {head}', flush=True)
        try:
            client = NotebookClient(nb, timeout=None, kernel_name='python3', resources={'metadata': {'path': str(work)}},
                                    on_cell_executed=_done)
        except Exception:
            client = NotebookClient(nb, timeout=None, kernel_name='python3', resources={'metadata': {'path': str(work)}})
        import threading
        plog = work / 'progress.log'
        if plog.exists(): plog.unlink()
        stop = threading.Event()
        def _tail():
            pos = 0
            while not stop.wait(15):
                try:
                    with open(plog, encoding='utf-8') as f:
                        f.seek(pos); new = f.read(); pos = f.tell()
                    for l in new.splitlines(): print(f'[{name}] {l}', flush=True)
                except OSError: pass
        th = threading.Thread(target=_tail, daemon=True); th.start()
        try: client.execute()
        finally: stop.set(); th.join(timeout=5)
    finally:
        nbformat.write(nb, work / 'executed_PRIVATE.ipynb')   # 환자 정보가 출력될 수 있어 압축에서 제외
        for k, v in old.items():
            if v is None: os.environ.pop(k, None)
            else: os.environ[k] = v
    return work / 'kcd_liver_v2_outputs', round((time.time() - t) / 60, 1)

def step(name, fn):
    t = time.time()
    try:
        out = fn(); REPORT['steps'][name] = {'status': 'ok', 'minutes': round((time.time() - t) / 60, 1), **(out or {})}
    except BaseException as ex:
        REPORT['steps'][name] = {'status': 'failed', 'minutes': round((time.time() - t) / 60, 1), 'error': f'{type(ex).__name__}: {ex}',
                                 'trace_tail': traceback.format_exc()[-1500:]}
        print('실패:', name, ex)
    print(name, REPORT['steps'][name]['status'], REPORT['steps'][name]['minutes'], '분')

In [ ]:
# ===== 1단계: 전체 MIMIC-IV 분석 노트북 =====
FULL_OUT = WORK / 'full' / 'kcd_liver_v2_outputs'
def s1():
    out, m = run_main('full', MIMIC_ROOT, 'mimic-iv', MAX_PATIENTS)
    flow = pd.read_csv(out / '04_cohort_flow.csv').to_dict('records')
    return {'notebook_minutes': m, 'cohort_flow': flow}
if RUN_STEPS['full']: step('1_full_notebook', s1)

In [ ]:
# ===== 2단계: 모델 개발·평가 (전체 MIMIC-IV, 공개 데모 환자 제외) =====
MODELS = WORK / 'models'
def s2():
    import importlib, scripts.develop_models as dm
    importlib.reload(dm)
    dm.main(['--out-dir', str(FULL_OUT), '--mimic-root', str(MIMIC_ROOT), '--source', 'mimic-iv', '--model-dir', str(MODELS)])
    R = json.loads((FULL_OUT / 'results_full.json').read_text(encoding='utf-8'))
    return {'agent3_eval_rows': R.get('agent3_development', {}).get('test_rows'), 'models_saved_locally': sorted(p.name for p in MODELS.glob('*.joblib'))}
if RUN_STEPS['develop']: step('2_develop_models', s2)

In [ ]:
# ===== 3단계: 공개 데모 100명에 학습된 모델을 적용해 시연 화면 생성 =====
def s3():
    if not (DEMO_CSV / 'hosp' / 'labevents.csv.gz').exists():
        args = [sys.executable, str(REPO / 'scripts' / 'fhir_demo_to_csv.py'), '--out', str(DEMO_CSV)]
        if DEMO_FHIR_DIR: args += ['--fhir-dir', str(DEMO_FHIR_DIR)]
        r = subprocess.run(args, capture_output=True, text=True); print(r.stdout[-500:], r.stderr[-500:])
        if r.returncode: raise RuntimeError('데모 데이터 준비 실패: 인터넷이 막혀 있으면 DEMO_FHIR_DIR를 지정하세요')
    out, m = run_main('demo', DEMO_CSV, 'mimic-iv-demo', None,
                      extra={'BUILD_AGENT3_REFERENCE=False': 'BUILD_AGENT3_REFERENCE=True'}, env={'KCD_MODEL_DIR': MODELS})
    if (FULL_OUT / '40_site_aggregates.json').exists():
        subprocess.run([sys.executable, str(REPO / 'scripts' / 'export_site_data.py'), '--out-dir', str(FULL_OUT), '--source', 'mimic-iv', '--aggregates-only'], check=True)
    return {'notebook_minutes': m, 'demo_cases': sorted(p.name for p in (REPO / 'site' / 'data' / 'cases').glob('*.json'))}
if RUN_STEPS['demo']: step('3_demo_replay', s3)

In [ ]:
# ===== 4단계: 돌려줄 파일 하나로 묶기 (집계와 공개 데모만) =====
FORBIDDEN_KEYS = {'subject_id', 'hadm_id', 'stay_id', 'labevent_id', 'case_id'}
def scan_ids(o, path='', hits=None):
    hits = [] if hits is None else hits
    if isinstance(o, dict):
        for k, v in o.items():
            if k in FORBIDDEN_KEYS: hits.append(path + '/' + k)
            scan_ids(v, path + '/' + str(k), hits)
    elif isinstance(o, list):
        for v in o[:2000]: scan_ids(v, path, hits)
    return hits
def s4():
    site = REPO / 'site' / 'data'
    if (FULL_OUT / 'results_full.json').exists():
        shutil.copy(FULL_OUT / 'results_full.json', site / 'results_full.json')
        hits = scan_ids(json.loads((site / 'results_full.json').read_text(encoding='utf-8')))
        if hits: raise RuntimeError(f'results_full.json에 환자 식별 키가 있습니다: {hits[:5]}')
    guard = subprocess.run([sys.executable, str(REPO / 'scripts' / 'check_public_data.py')], capture_output=True, text=True, cwd=REPO)
    print(guard.stdout)
    if guard.returncode: raise RuntimeError('공개 데이터 검사 실패: ' + guard.stdout)
    stamp = datetime.now().strftime('%Y%m%d_%H%M')
    zp = OUTBOX / f'kcd_return_package_{stamp}.zip'
    REPORT['finished_at'] = datetime.now().isoformat(timespec='seconds')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in site.rglob('*'):
            if p.is_file() and p.suffix in {'.json'}: z.write(p, 'site_data/' + str(p.relative_to(site)))
        for sub, pat in [('results_tables', '*'), ('figures', '*.png')]:
            for p in (FULL_OUT / sub).glob(pat) if (FULL_OUT / sub).exists() else []:
                z.write(p, f'full_{sub}/' + p.name)
        ID_COLS = {'subject_id', 'hadm_id', 'stay_id', 'case_id', 'labevent_id'}
        for p in sorted(FULL_OUT.glob('*.csv')):
            if any(s in p.name for s in ('_private', 'EVALUATION_ONLY')): continue
            try: cols = set(pd.read_csv(p, nrows=0).columns)
            except Exception: continue
            if not cols & ID_COLS: z.write(p, 'full_aggregates/' + p.name)
        for name in ['40_site_aggregates.json', '08b_clustering_check.json', 'run_config.json']:
            if (FULL_OUT / name).exists(): z.write(FULL_OUT / name, 'full_aggregates/' + name)
        if (REPO / 'docs' / 'VARIABLES.md').exists(): z.write(REPO / 'docs' / 'VARIABLES.md', 'VARIABLES.md')
        z.writestr('run_report.json', json.dumps(REPORT, ensure_ascii=False, indent=1, default=str))
    names = zipfile.ZipFile(zp).namelist()
    bad = [n for n in names if any(s in n for s in ('_private', '.sqlite', '.joblib', 'EVALUATION_ONLY', 'executed_'))]
    if bad: zp.unlink(); raise RuntimeError(f'금지 파일이 포함되어 압축을 지웠습니다: {bad}')
    print(f'완성: {zp}  ({zp.stat().st_size/1e6:.1f} MB, 파일 {len(names)}개)')
    return {'package': str(zp), 'files': len(names)}
if RUN_STEPS['package']: step('4_package', s4)
print(json.dumps({k: v.get('status') for k, v in REPORT['steps'].items()}, ensure_ascii=False))

In [ ]:
# ===== 5단계: 모델 4종 + 외부 검증 패키지. 결과 zip에도 모델과 모델 카드를 함께 넣습니다 =====
MODEL_FILES = ['agent3_gbm.joblib', 'prognosis_gbm.joblib', 'trajectory_qgbm.joblib', 'agent2_gbm.joblib',
               'agent3_gbm_2x.joblib', 'agent3_gbm_3x.joblib', 'agent3_gbm_4x.joblib', 'agent3_gbm_5x.joblib']
def s5():
    import joblib, sklearn
    R = json.loads((FULL_OUT / 'results_full.json').read_text(encoding='utf-8'))
    have = [n for n in MODEL_FILES if (MODELS / n).exists()]
    if not {'agent3_gbm.joblib', 'prognosis_gbm.joblib', 'trajectory_qgbm.joblib'} <= set(have):
        raise RuntimeError(f'필수 모델이 없습니다: {have}')
    spec = {}
    for n in have:
        b = joblib.load(MODELS / n)
        extra = set(b) - {'model', 'models', 'features', 'kind', 'classes', 'seed', 'trained_on', 'built_at', 'background', 'death_model', 'death_features'}
        if extra: raise RuntimeError(f'{n}: 예상하지 못한 내용 {extra}')
        spec[n] = {'kind': b.get('kind'), 'features': b.get('features'), 'classes': b.get('classes'),
                   'background': 'synthetic percentile rows' if 'background' in b else None}
    cm = FULL_OUT / '08c_cluster_models.joblib'
    stamp = datetime.now().strftime('%Y%m%d_%H%M')
    a3 = next((m for m in (R.get('agent3_performance') or {}).get('models', []) if m.get('model') == '그래디언트 부스팅'), {})
    card = f"""# KCD 2026 간기능 감시 모델 카드

- 학습 자료: MIMIC-IV v3.1 (공개 데모 100명 제외), 환자 단위 70/30 분할, 시드 {R['meta'].get('seed', 20261016)}
- 생성: {stamp} · 코드 버전 {REPORT.get('code_version', '-')}
- 실행 환경 (모델을 읽을 때 같은 버전 필요): python {platform.python_version()}, scikit-learn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}

## 정의
- 이상: ALT·ALP·총빌리루빈 중 하나라도 ≥ 2×ULN (검사별 참고범위 상한)
- 정상(Agent 3 판정 대상): 세 항목 모두 최근 24시간 내 측정, 모두 < 2×ULN
- 회복: ICU 퇴실 시점의 마지막 ALT·ALP·총빌리루빈이 모두 < 2×ULN (다시 재지 않은 항목이 있으면 판정 불가, 사망은 판정)
- 제외: ICU 입실 전 같은 입원 병동에서 이미 세 항목 중 하나라도 ≥ 2×ULN
- 추적 종료: ICU 퇴실 (병동 검사는 입원 시점부터 반영). 사망: ICU 사망 = ICU 퇴실 후 24시간 이내 사망 (원내 사망 시각, 없으면 사망 날짜)
- 조기 예측 목표: 24시간 안에 세 항목 중 하나라도 ≥ 5×ULN (2·3·4배 모델은 비교용)
- 모든 입력은 그 시각까지 결과가 보고된 값만 사용 (max(charttime, storetime))

## 모델
| # | 실행 시점 | 파일 | 출력 |
|---|---|---|---|
| 1 | 정상일 때 매시간 | agent3_gbm.joblib | 24시간 이내 이상 위험 (+ 그룹 Shapley, 합성 기준 행) |
| 2 | 첫 이상 시점 | prognosis_gbm.joblib | 예후 4경로 확률 (회복 판정 가능 사건) + ICU 사망 확률 (모든 사건) |
| 3 | 첫 이상 시점 | kcd_liver/rule_agent.py (학습 없음), cluster_models.joblib (보조) | 원인 10개 0/1 |
| 4 | 매시간, 상태 무관 | trajectory_qgbm.joblib | ALT·AST·ALP·TBIL 6·12·18·24시간 10·50·90% |

## 내부 검증 (MIMIC-IV 평가 집단)
- Agent 3 AUROC {a3.get('auroc')}, AUPRC {a3.get('auprc')}, Brier {a3.get('brier')}, 보정 기울기 {a3.get('cal_slope')}

## 사용 조건과 한계
- 연구용. 임상 결정에 쓰지 않습니다. 원인 10개는 진단이 아니라 근거 기반 후보입니다.
- 담낭염은 구조화 자료로 판단하지 않으며(영상·국소 소견 필요), 담관염은 TG18 '의심' 수준까지만 판정합니다.
- MIMIC-IV 자격 인증 자료로 학습한 모델입니다. 전달 전 PhysioNet 이용 약관과 기관 규정을 확인하세요.
- 외부 병원의 검사 itemid·단위가 다르면 대응표를 먼저 맞춰야 합니다.
"""
    zp = globals().get('OUTBOX', REPO) / f'kcd_model_package_{stamp}.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for n in have: z.write(MODELS / n, 'models/' + n)
        if cm.exists(): z.write(cm, 'models/cluster_models.joblib')
        for sub in ['kcd_liver', 'scripts', 'reference']:
            for p in (REPO / sub).rglob('*'):
                if p.is_file() and '__pycache__' not in p.parts: z.write(p, str(p.relative_to(REPO)))
        z.write(REPO / 'notebooks' / 'KCD_MIMICIV_liver_agents_v2.ipynb', 'notebooks/KCD_MIMICIV_liver_agents_v2.ipynb')
        z.write(REPO / 'notebooks' / 'KCD_EXTERNAL.ipynb', 'KCD_EXTERNAL.ipynb')
        if (REPO / 'docs' / 'VARIABLES.md').exists(): z.write(REPO / 'docs' / 'VARIABLES.md', 'VARIABLES.md')
        z.writestr('MODEL_CARD.md', card)
        z.writestr('feature_spec.json', json.dumps(spec, ensure_ascii=False, indent=1))
        z.writestr('requirements.txt', f"scikit-learn=={sklearn.__version__}\npandas=={pd.__version__}\nnumpy=={np.__version__}\njoblib\nnbformat\nnbclient\nipykernel\nmatplotlib\n")
    names = zipfile.ZipFile(zp).namelist()
    bad = [n for n in names if any(s in n for s in ('_private', '.sqlite', 'EVALUATION_ONLY', 'executed_', '.csv.gz'))]
    if bad: zp.unlink(); raise RuntimeError(f'금지 파일이 포함되어 압축을 지웠습니다: {bad}')
    rp = sorted(globals().get('OUTBOX', REPO).glob('kcd_return_package_*.zip'))
    if rp:
        with zipfile.ZipFile(rp[-1], 'a', zipfile.ZIP_DEFLATED) as z:
            for n in have: z.write(MODELS / n, 'models/' + n)
            if cm.exists(): z.write(cm, 'models/cluster_models.joblib')
            z.writestr('MODEL_CARD.md', card); z.writestr('feature_spec.json', json.dumps(spec, ensure_ascii=False, indent=1))
        print('결과 zip에 모델 추가:', rp[-1].name)
    print(f'모델 패키지 (외부 병원용): {zp} ({zp.stat().st_size/1e6:.1f} MB)')
    return {'model_package': str(zp), 'models': have + (['cluster_models.joblib'] if cm.exists() else [])}
if RUN_STEPS.get('package', True): step('5_model_package', s5)
print(json.dumps({k: v.get('status') for k, v in REPORT['steps'].items()}, ensure_ascii=False))

## 보낼 것

- **저에게 보낼 것**: `kcd_return_package_*.zip` 하나. 집계 결과·표·그림·시연 화면·실행 보고서·**학습된 모델 4종과 모델 카드**가 들어 있습니다. 실패한 단계가 있어도 `run_report.json`에 이유가 적혀 있으니 그대로 보내주세요.
- **외부 병원에 줄 것**: `kcd_model_package_*.zip`. 압축을 풀고 `KCD_EXTERNAL.ipynb`를 실행하면 `kcd_external_return_*.zip`(집계)이 나옵니다.
- **어디에도 보내지 않을 것**: `kcd_work/` 폴더 (원자료와 환자 단위 중간 파일).